In [1]:
# ── Cell 1: Install ──────────────────────────────────────────────────────────
!pip install -q catboost scikit-learn pandas numpy


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python3.14 -m pip install --upgrade pip


In [2]:
# ── Cell 2: Imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report

SEEDS    = [42, 7, 123]
N_SPLITS = 10
print('Libraries loaded.')

Libraries loaded.


In [3]:
# ── Cell 3: Load data ────────────────────────────────────────────────────────
TRAIN_DATA  = pd.read_csv('train-data.csv',  index_col='id')
TRAIN_LABEL = pd.read_csv('train-label.csv', index_col='id')
TEST_DATA   = pd.read_csv('test-data.csv',   index_col='id')

print(f'Train: {TRAIN_DATA.shape}, Test: {TEST_DATA.shape}')
print(TRAIN_LABEL['disorder'].value_counts().sort_index())

Train: (13249, 41), Test: (8834, 41)
disorder
0     389
1    2068
2    1090
3    3096
4      58
5    1700
6     813
7    2643
8      91
9    1301
Name: count, dtype: int64


In [4]:
# ── Cell 4: Preprocessing — a3 base + target encoding for institute cols ─────
#
# NEW vs a3:
#   - Extract district from institute_location string
#   - Keep insitute_name (27 unique values)
#   - Apply OOF target encoding for both: for each class, encode
#     the mean probability of being that class given the category
#     This gives the model 10 new columns per encoded feature
#     (one per disorder class) without leaking label info

def get_district(s):
    if pd.isna(s) or s == '-':
        return 'MISSING'
    m = re.search(r'\n(.+?),', str(s))
    return m.group(1).strip() if m else 'OTHER'


def preprocess_base(df):
    """Base preprocessing — same as a3 but keeps institute cols for encoding."""
    df = df.copy()

    # Extract district before dropping
    df['district'] = df['institute_location'].apply(get_district)
    df['institute_name_clean'] = df['insitute_name'].fillna('MISSING')

    drop_cols = [
        'first_name', 'last_name', 'insitute_name', 'institute_location',
        'test_1', 'test_2', 'test_3', 'test_4', 'test_5', 'treatment_consent'
    ]
    df = df.drop(columns=drop_cols)

    miss_cols = [
        'gender', 'maternal_defect', 'mother_age', 'father_age',
        'respiration', 'heart_rate', 'risk_level', 'place_birth',
        'folic_acid', 'maternal_illness', 'infertility_treatment',
        'problem_previous_pregnancies', 'abortion_cnt',
        'birth_defects', 'white_blood_cell_count', 'blood_test',
        'symptom_1', 'symptom_2', 'symptom_3', 'symptom_4', 'symptom_5'
    ]
    df['missing_count']      = df[miss_cols].isna().sum(axis=1)
    df['missing_parent_age'] = df['mother_age'].isna().astype(int) + df['father_age'].isna().astype(int)
    df['missing_symptoms']   = df[['symptom_1','symptom_2','symptom_3','symptom_4','symptom_5']].isna().sum(axis=1)
    df['missing_clinical']   = df[['respiration','heart_rate','risk_level','blood_test']].isna().sum(axis=1)

    for col in ['mother_age','father_age','maternal_defect','gender',
                'risk_level','heart_rate','respiration','abortion_cnt','white_blood_cell_count']:
        df[f'{col}_missing'] = df[col].isna().astype(int)

    binary_yn = [
        'mother_defect','father_defect','maternal_defect','paternal_defect',
        'alive','folic_acid','maternal_illness','infertility_treatment',
        'problem_previous_pregnancies',
        'symptom_1','symptom_2','symptom_3','symptom_4','symptom_5'
    ]
    for col in binary_yn:
        df[col] = df[col].map({'Y': 1, 'N': 0})

    df['respiration']   = df['respiration'].map({'A': 1, 'N': 0})
    df['heart_rate']    = df['heart_rate'].map({'A': 1, 'N': 0})
    df['risk_level']    = df['risk_level'].map({'H': 1, 'L': 0})
    df['place_birth']   = df['place_birth'].map({'I': 1, 'H': 0})
    df['birth_defects'] = df['birth_defects'].map({'S': 1, 'M': 2})
    df['gender']        = df['gender'].map({'M': 0, 'F': 1, 'A': 2})
    df['autopsy']       = df['autopsy'].map({'Y': 1, 'N': 0})
    for col in ['birth_asphyxia', 'radiation_exposure', 'substance_abuse']:
        df[col] = df[col].map({'Y': 1, 'N': 0, 'NR': 2})
    df['blood_test'] = df['blood_test'].map({'N': 0, 'I': 1, 'S': 2, 'A': 3})

    df['defect_sum']  = df[['mother_defect','father_defect','maternal_defect','paternal_defect']].sum(axis=1)
    df['symptom_sum'] = df[['symptom_1','symptom_2','symptom_3','symptom_4','symptom_5']].sum(axis=1)
    df['defect_x_symptom']     = df['defect_sum'] * df['symptom_sum']
    df['any_defect']           = (df['defect_sum'] > 0).astype(int)
    df['any_symptom']          = (df['symptom_sum'] > 0).astype(int)
    df['high_symptom']         = (df['symptom_sum'] >= 4).astype(int)
    df['all_defects']          = (df['defect_sum'] == 4).astype(int)
    df['parent_age_gap']       = (df['father_age'] - df['mother_age']).abs()
    df['symptom_defect_ratio'] = df['symptom_sum'] / (df['defect_sum'] + 1)
    df['s4_and_s5']    = ((df['symptom_4'] == 1) & (df['symptom_5'] == 1)).astype(int)
    df['no_s4_s5']     = ((df['symptom_4'] == 0) & (df['symptom_5'] == 0)).astype(int)
    df['late_vs_early']= (df['symptom_4'].fillna(0) + df['symptom_5'].fillna(0)
                         - df['symptom_1'].fillna(0) - df['symptom_2'].fillna(0))
    df['weighted_sym'] = (df['symptom_1'].fillna(0)*1 + df['symptom_2'].fillna(0)*1 +
                          df['symptom_3'].fillna(0)*1 + df['symptom_4'].fillna(0)*2 +
                          df['symptom_5'].fillna(0)*2)
    df['both_parents_defect'] = ((df['mother_defect'] == 1) & (df['father_defect'] == 1)).astype(int)
    df['no_parent_defect']    = ((df['mother_defect'] == 0) & (df['father_defect'] == 0)).astype(int)

    return df


X_train_raw = preprocess_base(TRAIN_DATA)
X_test_raw  = preprocess_base(TEST_DATA)
y_train     = TRAIN_LABEL['disorder'].values
print(f'Base features: {X_train_raw.shape[1]}')

Base features: 61


In [5]:
# ── Cell 5: OOF Target Encoding ───────────────────────────────────────────────
# For each categorical col (institute_name, district):
#   For each class c in 0..9:
#     encode[row] = P(label==c | category) computed on other folds only
# This gives 2 cols × 10 classes = 20 new features
# Test set uses global mean from full training set (no leakage risk there)

N_CLASSES = 10
TE_COLS   = ['institute_name_clean', 'district']
TE_FOLDS  = 5

def oof_target_encode(X_tr, y_tr, X_te, cat_col, n_classes=10, n_folds=5, seed=42):
    """OOF target encoding — returns (train_enc, test_enc) arrays of shape (n, n_classes)."""
    train_enc = np.zeros((len(X_tr), n_classes))
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)

    # Global mean per category (used as fallback and for test)
    global_mean = np.zeros((X_tr[cat_col].nunique() + 1, n_classes))
    # Build global stats
    cat_vals = X_tr[cat_col].fillna('MISSING').astype(str)
    categories = cat_vals.unique()
    global_stats = {}
    for cat in categories:
        mask = cat_vals == cat
        y_cat = y_tr[mask]
        probs = np.zeros(n_classes)
        for c in range(n_classes):
            probs[c] = (y_cat == c).mean() if len(y_cat) > 0 else 1/n_classes
        global_stats[cat] = probs

    # OOF encoding
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_tr, y_tr)):
        cat_tr  = cat_vals.iloc[tr_idx]
        y_tr_f  = y_tr[tr_idx]
        cat_val = cat_vals.iloc[val_idx]

        fold_stats = {}
        for cat in cat_tr.unique():
            mask = cat_tr == cat
            y_cat = y_tr_f[mask]
            probs = np.zeros(n_classes)
            for c in range(n_classes):
                probs[c] = (y_cat == c).mean() if len(y_cat) > 0 else 1/n_classes
            fold_stats[cat] = probs

        for i, cat in enumerate(cat_val):
            train_enc[val_idx[i]] = fold_stats.get(cat, global_stats.get(cat, np.full(n_classes, 1/n_classes)))

    # Test encoding uses global stats
    cat_te = X_te[cat_col].fillna('MISSING').astype(str)
    test_enc = np.zeros((len(X_te), n_classes))
    for i, cat in enumerate(cat_te):
        test_enc[i] = global_stats.get(cat, np.full(n_classes, 1/n_classes))

    return train_enc, test_enc


print('Computing OOF target encodings...')
te_train_parts = []
te_test_parts  = []

for col in TE_COLS:
    tr_enc, te_enc = oof_target_encode(X_train_raw, y_train, X_test_raw, col,
                                        n_classes=N_CLASSES, n_folds=TE_FOLDS)
    col_names = [f'te_{col}_cls{c}' for c in range(N_CLASSES)]
    te_train_parts.append(pd.DataFrame(tr_enc, columns=col_names))
    te_test_parts.append(pd.DataFrame(te_enc,  columns=col_names))
    print(f'  {col}: done')

# Drop raw string columns and append encoded ones
X_train = X_train_raw.drop(columns=TE_COLS).reset_index(drop=True)
X_test  = X_test_raw.drop(columns=TE_COLS).reset_index(drop=True)

X_train = pd.concat([X_train] + te_train_parts, axis=1)
X_test  = pd.concat([X_test]  + te_test_parts,  axis=1)

print(f'\nFeatures after target encoding: {X_train.shape[1]}  (+{len(TE_COLS)*N_CLASSES} TE cols)')

Computing OOF target encodings...
  institute_name_clean: done
  district: done

Features after target encoding: 79  (+20 TE cols)


In [6]:
# ── Cell 6: Class weights ─────────────────────────────────────────────────────
class_counts  = np.bincount(y_train)
class_weights = len(y_train) / (10 * class_counts)
print('Class weights:')
for i, (n, w) in enumerate(zip(class_counts, class_weights)):
    print(f'  Class {i}: weight={w:.3f}  (n={n})')

Class weights:
  Class 0: weight=3.406  (n=389)
  Class 1: weight=0.641  (n=2068)
  Class 2: weight=1.216  (n=1090)
  Class 3: weight=0.428  (n=3096)
  Class 4: weight=22.843  (n=58)
  Class 5: weight=0.779  (n=1700)
  Class 6: weight=1.630  (n=813)
  Class 7: weight=0.501  (n=2643)
  Class 8: weight=14.559  (n=91)
  Class 9: weight=1.018  (n=1301)


In [7]:
# ── Cell 7: Stage 1 — Binary classifier: class 9 vs rest ─────────────────────
# Class 9 (확인안됨) is nearly identical to class 3 in feature space
# A dedicated binary model focusing only on this distinction
# should learn subtler boundaries than the 10-class model

print('STAGE 1: Binary classifier — class 9 vs rest')
print('=' * 55)

y_binary = (y_train == 9).astype(int)  # 1 = class 9, 0 = everything else
print(f'Class 9: {y_binary.sum()} rows, Other: {(y_binary==0).sum()} rows')

# Balanced weights for binary task
n_pos = y_binary.sum()
n_neg = len(y_binary) - n_pos
bin_class_weights = [1.0, n_neg / n_pos]  # [weight_0, weight_1]
print(f'Binary weights: class_0={bin_class_weights[0]:.2f}, class_9={bin_class_weights[1]:.2f}')

stage1_oof_proba  = np.zeros(len(y_train))   # P(class_9) for each train row
stage1_test_proba = np.zeros(len(X_test))    # P(class_9) for each test row

for SEED in SEEDS:
    print(f"\n  SEED={SEED}")
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    oof_p   = np.zeros(len(y_train))
    test_p  = np.zeros(len(X_test))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_binary)):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_binary[tr_idx],     y_binary[val_idx]

        model = CatBoostClassifier(
            iterations=2000, learning_rate=0.03, depth=5,
            l2_leaf_reg=5,   # extra regularization for binary task
            class_weights=bin_class_weights,
            early_stopping_rounds=100,
            eval_metric='AUC',
            random_seed=SEED, verbose=0, thread_count=-1,
        )
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)

        val_proba = model.predict_proba(X_val)[:, 1]
        oof_p[val_idx] += val_proba
        test_p         += model.predict_proba(X_test)[:, 1] / N_SPLITS

        # Use 0.5 threshold to get recall of class 9
        val_pred = (val_proba > 0.5).astype(int)
        fold_scores.append(balanced_accuracy_score(y_val, val_pred))

    print(f'    mean fold BA: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')
    stage1_oof_proba  += oof_p / len(SEEDS)
    stage1_test_proba += test_p / len(SEEDS)

# Evaluate at various thresholds
print('\nStage 1 threshold analysis (P(class9) > threshold → predict class 9):')
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    preds = (stage1_oof_proba > t).astype(int)
    tp = ((preds == 1) & (y_binary == 1)).sum()
    fp = ((preds == 1) & (y_binary == 0)).sum()
    recall_9 = tp / y_binary.sum()
    precision_9 = tp / (tp + fp) if (tp + fp) > 0 else 0
    print(f'  t={t:.1f}: recall_9={recall_9:.3f}  precision_9={precision_9:.3f}  predicted_9={preds.sum()}')

STAGE 1: Binary classifier — class 9 vs rest
Class 9: 1301 rows, Other: 11948 rows
Binary weights: class_0=1.00, class_9=9.18

  SEED=42
    mean fold BA: 0.6700 ± 0.0101

  SEED=7
    mean fold BA: 0.6632 ± 0.0167

  SEED=123
    mean fold BA: 0.6694 ± 0.0075

Stage 1 threshold analysis (P(class9) > threshold → predict class 9):
  t=0.3: recall_9=0.986  precision_9=0.135  predicted_9=9523
  t=0.4: recall_9=0.984  precision_9=0.142  predicted_9=8993
  t=0.5: recall_9=0.958  precision_9=0.143  predicted_9=8750
  t=0.6: recall_9=0.015  precision_9=0.202  predicted_9=94
  t=0.7: recall_9=0.000  precision_9=0.000  predicted_9=0


In [8]:
# ── Cell 8: Stage 2 — 10-class model on full features ────────────────────────
print('STAGE 2: 10-class CatBoost (same as best run)')
print('=' * 55)

# Add Stage 1 probability as a feature for Stage 2
X_train_s2 = X_train.copy()
X_test_s2  = X_test.copy()
X_train_s2['p_class9_stage1'] = stage1_oof_proba   # OOF proba — no leakage
X_test_s2['p_class9_stage1']  = stage1_test_proba

stage2_oof_proba  = np.zeros((len(y_train), 10))
stage2_test_preds = np.zeros((len(X_test), 10))

for SEED in SEEDS:
    print(f"\n{'='*40} SEED={SEED} {'='*40}")
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    oof_proba   = np.zeros((len(y_train), 10))
    test_preds  = np.zeros((len(X_test), 10))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_s2, y_train)):
        X_tr, X_val = X_train_s2.iloc[tr_idx], X_train_s2.iloc[val_idx]
        y_tr, y_val = y_train[tr_idx],          y_train[val_idx]

        model = CatBoostClassifier(
            iterations=2000, learning_rate=0.03, depth=6, l2_leaf_reg=3,
            class_weights=class_weights, early_stopping_rounds=100,
            eval_metric='Accuracy', random_seed=SEED, verbose=0, thread_count=-1,
        )
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)

        val_proba = model.predict_proba(X_val)
        score     = balanced_accuracy_score(y_val, np.argmax(val_proba, axis=1))
        fold_scores.append(score)
        print(f'  Fold {fold+1:2d}: BA={score:.4f}  best_iter={model.best_iteration_}')
        oof_proba[val_idx] += val_proba
        test_preds         += model.predict_proba(X_test_s2) / N_SPLITS

    oof_score = balanced_accuracy_score(y_train, np.argmax(oof_proba, axis=1))
    print(f'  OOF BA (seed={SEED}): {oof_score:.4f} | mean={np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')
    stage2_oof_proba  += oof_proba  / len(SEEDS)
    stage2_test_preds += test_preds / len(SEEDS)

stage2_oof_score = balanced_accuracy_score(y_train, np.argmax(stage2_oof_proba, axis=1))
print(f'\nSTAGE 2 OOF BA: {stage2_oof_score:.4f}  (best so far: 0.3933)')

STAGE 2: 10-class CatBoost (same as best run)

======================================== SEED=42 ========================================
  Fold  1: BA=0.3723  best_iter=68
  Fold  2: BA=0.3961  best_iter=12
  Fold  3: BA=0.4028  best_iter=175
  Fold  4: BA=0.3707  best_iter=102
  Fold  5: BA=0.4033  best_iter=70
  Fold  6: BA=0.4381  best_iter=156
  Fold  7: BA=0.3340  best_iter=29
  Fold  8: BA=0.3678  best_iter=9
  Fold  9: BA=0.4196  best_iter=55
  Fold 10: BA=0.4213  best_iter=47
  OOF BA (seed=42): 0.3924 | mean=0.3926 ± 0.0296

======================================== SEED=7 ========================================
  Fold  1: BA=0.3914  best_iter=79
  Fold  2: BA=0.4196  best_iter=69
  Fold  3: BA=0.4004  best_iter=109
  Fold  4: BA=0.4173  best_iter=283
  Fold  5: BA=0.3449  best_iter=34
  Fold  6: BA=0.4030  best_iter=17
  Fold  7: BA=0.3720  best_iter=17
  Fold  8: BA=0.3926  best_iter=53
  Fold  9: BA=0.4104  best_iter=215
  Fold 10: BA=0.3969  best_iter=64
  OOF BA (seed=7):

In [9]:
# ── Cell 9: Per-class recall comparison ──────────────────────────────────────
disorder_names = {
    0:'레베르시', 1:'낭포성섬유증', 2:'당뇨', 3:'리증후군', 4:'암',
    5:'테이-삭스', 6:'혈색소침착증', 7:'사립체근병종', 8:'알츠하이머', 9:'확인안됨'
}
best_recall = {0:0.314, 1:0.390, 2:0.270, 3:0.348, 4:0.793,
               5:0.339, 6:0.498, 7:0.238, 8:0.604, 9:0.139}

oof_labels = np.argmax(stage2_oof_proba, axis=1)
report     = classification_report(y_train, oof_labels, output_dict=True)

print(f'OOF BA: {stage2_oof_score:.4f}  (best LB run was 0.3933 OOF → 0.37127 LB)\n')
print(f'{"Class":<5} {"Name":<16} {"Best run":>10} {"Now":>8} {"Δ":>7}')
print('-' * 52)
for cls in range(10):
    r     = report[str(cls)]['recall']
    r_old = best_recall[cls]
    delta = r - r_old
    flag  = ' ← up' if delta > 0.02 else ' ← LOW' if r < 0.3 else ''
    print(f'{cls:<5} {disorder_names[cls]:<16} {r_old:>10.3f} {r:>8.3f} {delta:>+7.3f}{flag}')

OOF BA: 0.3756  (best LB run was 0.3933 OOF → 0.37127 LB)

Class Name               Best run      Now       Δ
----------------------------------------------------
0     레베르시                  0.314    0.308  -0.006
1     낭포성섬유증                0.390    0.401  +0.011
2     당뇨                    0.270    0.252  -0.018 ← LOW
3     리증후군                  0.348    0.349  +0.001
4     암                     0.793    0.741  -0.052
5     테이-삭스                 0.339    0.328  -0.011
6     혈색소침착증                0.498    0.466  -0.032
7     사립체근병종                0.238    0.244  +0.006 ← LOW
8     알츠하이머                 0.604    0.571  -0.033
9     확인안됨                  0.139    0.095  -0.044 ← LOW


In [10]:
# ── Cell 10: Save submission ──────────────────────────────────────────────────
final_preds = np.argmax(stage2_test_preds, axis=1)
submission  = pd.DataFrame({
    'id'      : TEST_DATA.index,
    'disorder': final_preds
}).set_index('id')
submission.to_csv('submission_2stage_te.csv')

print('Saved: submission_2stage_te.csv')
print(f'Shape: {submission.shape}')
print('\nPrediction distribution:')
print(submission['disorder'].value_counts().sort_index())
print(f'\nOOF BA: {stage2_oof_score:.4f}')
print(f'Submit if OOF > 0.3933 (current best run OOF)')
print(f'Gap rule: expect LB ≈ OOF - 0.022')

Saved: submission_2stage_te.csv
Shape: (8834, 1)

Prediction distribution:
disorder
0     340
1    1482
2     570
3    1569
4     308
5    1315
6    1054
7    1183
8     298
9     715
Name: count, dtype: int64

OOF BA: 0.3756
Submit if OOF > 0.3933 (current best run OOF)
Gap rule: expect LB ≈ OOF - 0.022
